# 03 - Kinematic feature clustering

**Purpose**: understand how kinematic features relate to each other.
Cluster them by pairwise correlation, then view the same clusters
projected into PCA space.

**Input**: raw ``data.AKDdf`` restricted to contacted reaches, using
every numeric non-metadata column (including unit duplicates) so the
dendrogram reveals which measurements are functionally redundant.

Intuition target: confirm that ``prefer_calibrated_units`` is dropping
the right duplicates (same measurement in different units should
cluster together), and see which non-duplicate features carry
overlapping information.


In [ ]:
# parameters
N_CLUSTERS = 8        # Tune to where the dendrogram visually splits
FIGSIZE_DENDRO = (16, 6)
FIGSIZE_2D = (12, 10)


In [ ]:
import numpy as np                                                       # numerical arrays
import pandas as pd                                                      # dataframes
import matplotlib.pyplot as plt                                          # 2D plotting
import plotly.express as px                                              # interactive plots; used here for the 3D scatter at the end
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster        # hierarchical clustering: linkage builds the merge tree, dendrogram draws it, fcluster cuts it into a target cluster count
from scipy.spatial.distance import pdist                                 # condensed pairwise-distance vector that linkage() expects
from sklearn.preprocessing import StandardScaler                         # z-scorer
from sklearn.decomposition import PCA                                    # principal components

from endpoint_ck_analysis.config import EXAMPLE_OUTPUT_DIR, METADATA_COLS # PNG output dir + the set of column names that are NOT kinematic features (used to subtract metadata from numeric columns)
from endpoint_ck_analysis.data_loader import load_all                     # one-shot data loader
from endpoint_ck_analysis.helpers.plotting import stamp_version           # figure version footer

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)                    # ensure output folder exists
data = load_all()                                                        # uses cache from 00_setup


## 1. Build the feature matrix

Contacted reaches only. Every numeric non-metadata column (including
unit duplicates) is retained so the dendrogram can reveal redundancy.


In [ ]:
contacted = data.AKDdf[data.AKDdf['contact_group'] == 'contacted']                                # boolean filter: keep only reaches where the mouse touched the pellet (uncontacted reaches don't have meaningful kinematics)
kine_cols = [c for c in contacted.select_dtypes(include='number').columns if c not in METADATA_COLS] # list comprehension: pick numeric columns that are NOT in METADATA_COLS (METADATA_COLS lists ID/index columns we don't want to treat as features)
feature_matrix = contacted[kine_cols].fillna(0)  # rows = one reach each, columns = one kinematic feature each; fillna(0) replaces missing values so PCA/clustering doesn't error

# Convert correlation to distance (highly correlated -> short distance).
# |r| treats negative correlations as similarity (same info, flipped sign).
corr = feature_matrix.corr()                                                                       # pandas .corr() returns a square (features x features) Pearson correlation matrix
dist = 1 - corr.abs()                                                                              # absolute value because a feature correlated -1 with another carries the same information; flip the sign and they're identical

link = linkage(pdist(dist), method='ward')  # pdist condenses the square dist matrix into the upper-triangle vector linkage() needs; Ward = a hierarchical clustering criterion that minimizes within-cluster variance at each merge

cluster_ids = fcluster(link, t=N_CLUSTERS, criterion='maxclust')                                   # cut the merge tree into exactly N_CLUSTERS flat clusters; criterion='maxclust' tells fcluster that t is a target cluster count (not a distance threshold)
feature_clusters = pd.Series(cluster_ids, index=corr.columns, name='cluster')                      # wrap the integer label array as a Series indexed by feature name so we can join it with PC coordinates downstream


## 2. Dendrogram


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE_DENDRO)                                                  # one figure, one axis; FIGSIZE_DENDRO is (16, 6) by default in the parameters cell
dendrogram(link, labels=corr.columns.tolist(), leaf_rotation=90, ax=ax)                          # draw the merge tree; leaf_rotation=90 prints feature names vertically so they don't overlap
ax.set_ylabel('Clustering distance')                                                             # height in the dendrogram = how dissimilar two clusters were at the moment they merged
ax.set_title(f'Kinematic feature dendrogram (contacted reaches; cut at {N_CLUSTERS} clusters)')
plt.tight_layout()                                                                               # prevent label overlap
stamp_version(fig, label='03 dendrogram')
plt.savefig(EXAMPLE_OUTPUT_DIR / '03_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()


**What this figure shows.** Hierarchical clustering of kinematic
features by how much they correlate with each other across reaches.
Height in the tree = clustering distance (1 - |Pearson r|, so
height 0 means perfect correlation, height 1 means uncorrelated).
Features are joined into clusters from bottom up; horizontal cuts
across the tree at higher distances yield fewer, broader clusters.

**What different patterns mean.**
- Pairs of features at distance ~0 = essentially the same
  measurement. *Example:* `max_extent_mm` and `max_extent_pixels`
  should sit at distance ~0; this is the unit-deduplication pass
  validating itself.
- Tight cluster of 3-4 features at low height = those features
  carry overlapping information; the paper can pick one
  representative per cluster without losing biology.
- Features sitting alone at the right edge until very late merges
  = they carry independent information not captured by anything
  else. *Example:* if `trajectory_smoothness` only joins the rest
  of the tree at distance ~0.8, no other feature in the dataset
  is restating it -- it's a unique kinematic axis the paper
  should report.
- Big chunks of the dendrogram at similar heights = the kinematic
  feature space has clear functional groupings (e.g., velocity
  features clustering together, separate from path-shape features).

**Scientific reading.** Use this to decide which features to keep
in the paper. Within a tight cluster, picking one representative
is defensible. Across distant clusters, all of them deserve
reporting because they each capture something independent.


## 3. PCA on the same feature matrix

Fit a feature-space PCA (features as the items, reaches as the
observations) so we can plot features in 2D/3D.


In [ ]:
X_scaled = StandardScaler().fit_transform(feature_matrix)                # z-score every column so PC distances are unit-agnostic
pca_feat = PCA(n_components=3)                                           # request 3 components -- enough to position features in 3D
pca_feat.fit(X_scaled)                                                   # fit only (no .transform here -- we want the components matrix, not subject scores)

loadings_xyz = pd.DataFrame(                                             # build a feature-coordinate table from the components matrix
    pca_feat.components_.T,                                              # .components_ is shape (n_components, n_features); .T flips so rows=features, cols=PCs
    index=feature_matrix.columns,                                        # keep the feature names as row labels
    columns=['PC1', 'PC2', 'PC3'],
)
loadings_xyz['cluster'] = feature_clusters                               # attach each feature's cluster ID as a column for color coding below


## 4. 2D view with one label per cluster

Labels land on the feature closest to each cluster's 2D centroid to
keep the plot legible.


In [ ]:
representatives = []                                                                     # list of feature names, one per cluster, that we'll use as visible labels on the scatter
for cid, group in loadings_xyz.groupby('cluster'):                                       # iterate cluster by cluster; group is a DataFrame of features in this cluster
    centroid = group[['PC1', 'PC2']].mean()                                              # 2D centroid of the cluster (mean PC1 and mean PC2)
    dists = ((group[['PC1', 'PC2']] - centroid) ** 2).sum(axis=1)                        # squared Euclidean distance from each feature to the centroid (no sqrt needed since we just want the minimum)
    representatives.append(dists.idxmin())                                               # idxmin returns the feature name with the smallest distance -> closest to the centroid

fig, ax = plt.subplots(figsize=FIGSIZE_2D)                                               # one figure, one axis
scatter = ax.scatter(                                                                    # 2D scatter
    loadings_xyz['PC1'], loadings_xyz['PC2'],                                            # x, y coords
    c=loadings_xyz['cluster'], cmap='tab10', alpha=0.8, s=60,                            # color by cluster ID using tab10 (10 distinct colors); s = marker size in points
)
for feature in representatives:                                                          # one annotation per cluster representative
    row = loadings_xyz.loc[feature]
    ax.annotate(feature, (row['PC1'], row['PC2']), fontsize=9, fontweight='bold')        # label only the representatives so the plot stays readable
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')                               # reference line at PC2=0
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')                               # reference line at PC1=0
ax.set_xlabel(f"PC1 ({pca_feat.explained_variance_ratio_[0]:.1%} variance)")             # axis label includes variance fraction; .1% formats as percentage with 1 decimal
ax.set_ylabel(f"PC2 ({pca_feat.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title(f'Kinematic features in PC1-PC2 space ({N_CLUSTERS} clusters)')
plt.colorbar(scatter, ax=ax, label='Cluster ID', ticks=range(1, N_CLUSTERS + 1))         # colorbar for the cluster legend; ticks=1..K (skip 0 since cluster IDs start at 1)
plt.tight_layout()
stamp_version(fig, label='03 2D PCA')
plt.savefig(EXAMPLE_OUTPUT_DIR / '03_feature_clusters_2d.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nCluster membership ({N_CLUSTERS} clusters):')                                 # print which features ended up in which cluster -- crucial for interpreting the plot
for cid, group in loadings_xyz.groupby('cluster'):
    print(f'  Cluster {cid} ({len(group)} features): {sorted(group.index.tolist())}')


**What this figure + printout show.** The 2D scatter is each feature
positioned in PC1-PC2 space (using a feature-space PCA, which is
different from the per-phase PCA in notebook 02). Color = cluster
ID from the dendrogram cut. The bold-labeled feature in each
cluster is the one closest to its cluster's centroid -- a
reasonable representative. The printout below the figure lists
every feature's cluster assignment.

**What different patterns mean.**
- Tight, well-separated clusters in 2D = the dendrogram structure
  survives the projection to 2D; the clustering is geometrically
  coherent in feature space.
- Spread or overlapping clusters in 2D = the clustering forced
  structure that the 2D PC view doesn't see clearly. Could mean
  PC1+PC2 don't capture the relevant feature relationships --
  consider the 3D view.
- A cluster with many features piled on each other in 2D = those
  features are tightly equivalent; the bold representative is a
  good stand-in for the whole bundle.
- A cluster spread out along PC1 or PC2 = the cluster's features
  cover a range; the centroid is a less faithful representative.

**Scientific reading.** The printout is the practical takeaway --
it tells the paper "if you want to summarize the kinematics with
N axes, here are the natural N groups, and here are the features
in each one." The 2D view is a sanity check on cluster geometry.


## 5. 3D view (plotly -- interactive)

Hover a dot to see its feature name; drag to rotate.
Not saved as a PNG because the interactivity is the point.


In [ ]:
fig3d = px.scatter_3d(                                                                # plotly's interactive 3D scatter (matplotlib's 3D doesn't get hover-labels)
    loadings_xyz.reset_index().rename(columns={'index': 'feature'}),                  # reset_index() pulls the feature names off the index into a column; rename so plotly knows what to call it
    x='PC1', y='PC2', z='PC3',                                                        # axis assignments
    hover_name='feature',                                                             # what shows in the tooltip when you hover a dot
    color=loadings_xyz['cluster'].astype(str).values,                                 # cast cluster IDs to strings so plotly treats them as discrete categories (not a continuous gradient)
    color_discrete_sequence=px.colors.qualitative.T10,                                # plotly's "T10" qualitative palette = the same 10 colors as matplotlib's tab10, so 2D and 3D views are visually consistent
    title='Kinematic features in PC1-PC2-PC3 space (colored by cluster)',
    labels={                                                                          # axis-label overrides: include variance fraction in each label
        'PC1': f"PC1 ({pca_feat.explained_variance_ratio_[0]:.1%})",
        'PC2': f"PC2 ({pca_feat.explained_variance_ratio_[1]:.1%})",
        'PC3': f"PC3 ({pca_feat.explained_variance_ratio_[2]:.1%})",
        'color': 'Cluster',                                                           # legend title for the color dimension
    },
)
fig3d.update_traces(marker=dict(size=5, opacity=0.85))                                # tweak marker visuals after plot construction; size in pixels, opacity 0..1
fig3d.show()                                                                          # render inline; rotate the plot by drag, hover a dot for its label


**What this figure shows.** Same data as the 2D scatter, plus PC3 as
a third axis. Drag to rotate; hover any dot to see the feature
name. Color matches the 2D plot's cluster colors.

**When to use this view.** When clusters look ambiguous in 2D
because PC1+PC2 alone don't separate them, the third dimension
often does. *Example:* if cluster A and cluster B overlap in the
2D scatter but pull apart along PC3, the cluster structure is
real -- it just doesn't project cleanly onto the first two
principal axes.

**Reading the rotation.** Watch which dots stay close together as
you rotate; those are genuinely similar features regardless of
viewing angle. Dots that swap apparent neighbors as the camera
moves are positioned in 3D such that 2D projections can be
misleading.
